# Fed 商业票据 A2/P2−AA 30日等级利差：来源预检

两次定额只读请求、原字节先入NAS，再按Qlib真实日历保守5会话滞后。无未来SOXX标签、模型或账户；详见plan.json。

In [1]:
import hashlib
import io
import json
import os
import shlex
import subprocess
import tarfile
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _settings, _ssh, archive_nas_recorder, nas_api_call

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/fed_cp_rating_spread_source_20261023'
plan=json.loads((card/'plan.json').read_text())
assert plan['frozen'] and len(plan['requests'])==plan['budget']['HTTP_GETs']==2 and not (card/'run_state.json').exists()
assert plan['budget']=={'HTTP_GETs':2,'redirects':0,'retries':0,'max_bytes_each':5000000,'Qlib_D_calendar_calls':1,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'new_Qlib_Recorders':1}
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha(root/plan['provider']['calendar_uri']/'qlib_store_manifest.json')==plan['provider']['calendar_manifest_sha256']
snap=root/'data/snapshots/fed_cp_rating_spread_source_20261023';snap.mkdir(parents=True,exist_ok=True)
state={'status':'raw_capture_attempting','plan_sha256':sha(card/'plan.json'),'HTTP_GET_attempts':0,'Qlib_D_calendar_calls':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
atomic_json(state,card/'run_state.json')
responses=[]
for req in plan['requests']:
    series=req['series'];body=snap/(series+'.csv');headers=snap/(series+'.headers.txt')
    assert not body.exists() and not headers.exists(), 'frozen source capture already exists'
    started=datetime.now(timezone.utc).isoformat();state['HTTP_GET_attempts']+=1;atomic_json(state,card/'run_state.json')
    run=subprocess.run(['curl','-sS','--connect-timeout','5','--max-time','30','--max-redirs','0','-H','Accept: text/csv','-A','soxl-qlib-research/1.0','-D',str(headers),'-o',str(body),'-w','%{http_code}|%{url_effective}',req['url']],capture_output=True,text=True)
    finished=datetime.now(timezone.utc).isoformat()
    parts=run.stdout.strip().split('|',1)
    status=int(parts[0]) if parts and parts[0].isdigit() else 0
    final_url=parts[1] if len(parts)>1 else ''
    if not body.exists():body.write_bytes(b'')
    if not headers.exists():headers.write_bytes(b'')
    responses.append({'series':series,'url':req['url'],'request_started_utc':started,'response_finished_utc':finished,'http_status':status,'curl_exit_code':run.returncode,'final_url':final_url,'body_sha256':sha(body),'body_bytes':body.stat().st_size,'headers_sha256':sha(headers),'headers_bytes':headers.stat().st_size,'content_type':next((line.strip() for line in headers.read_text(errors='replace').splitlines() if line.lower().startswith('content-type:')),None)})
assert len(responses)==2 and state['HTTP_GET_attempts']==2
atomic_json({'status':'two_exact_requests_recorded','plan_sha256':state['plan_sha256'],'responses':responses,'market_outcome_reads':0,'model_fits':0,'native_accounts':0},card/'retrieval_receipt.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='fed-cp-original-') as tmp:
    archive=Path(tmp)/'sources.tar.gz'
    with tarfile.open(archive,'w:gz') as output:
        for req in plan['requests']:
            for suffix in ('.csv','.headers.txt'):
                p=snap/(req['series']+suffix);output.add(p,arcname=p.name)
    digest=sha(archive)
    receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')
    evidence=receipt['artifact']
    assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
    path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
    remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(path)).split()[0]
    assert remote==digest
    with tempfile.TemporaryDirectory(prefix='fed-cp-restore-') as restore:
        copy=Path(restore)/'sources.tar.gz'
        with copy.open('wb') as out:subprocess.run(['ssh','-o','BatchMode=yes','-p',str(settings['ssh']['port']),settings['ssh']['host'],'cat',path],stdout=out,stderr=subprocess.PIPE,check=True)
        assert sha(copy)==digest
        with tarfile.open(copy,'r:gz') as inp:
            for req in plan['requests']:
                for suffix in ('.csv','.headers.txt'):
                    p=snap/(req['series']+suffix);raw=inp.extractfile(p.name).read();assert len(raw)==p.stat().st_size and hashlib.sha256(raw).hexdigest()==sha(p)
source_archive={'status':'exact_received_bytes_NAS_scoped_restore_verified','archive_sha256':digest,'archive_bytes':evidence['size_bytes'],'nas_path':path,'evidence':evidence,'entries':[{'path':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(snap.iterdir()) if p.is_file()],'scoped_members_verified':4}
atomic_json(source_archive,card/'source_archive_receipt.json')
transport_ok=all(x['http_status']==200 and x['curl_exit_code']==0 and x['final_url']==x['url'] and 0<x['body_bytes']<=plan['budget']['max_bytes_each'] for x in responses)
if not transport_ok:
    summary={'experiment':plan['experiment'],'decision':'needs_exact_Fed_CP_source_transport_no_fit','objective_achieved':False,'plan_sha256':state['plan_sha256'],'retrieval':'retrieval_receipt.json','source_archive':'source_archive_receipt.json','transport_ok':False,'new_HTTP_GETs':2,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
    atomic_json(summary,card/'summary.json')
    state['status']='transport_failed_no_mapping';atomic_json(state,card/'run_state.json')
    (card/'conclusion.md').write_text('# Fed商业票据信用等级利差：来源传输门\n\n两条冻结官方转发CSV各请求一次，原响应和报头字节已哈希并归档NAS、限定恢复；至少一条未以原URL返回有效HTTP200，所以没有解析因子、读取SOXX结果、拟合或账户。具体请求状态见retrieval_receipt.json。决定：needs_exact_Fed_CP_source_transport_no_fit。\n')
    print(summary['decision'],[(x['series'],x['http_status'],x['body_bytes']) for x in responses])
else:
    parsed={}
    for req in plan['requests']:
        series=req['series'];text=(snap/(series+'.csv')).read_bytes()
        d=pd.read_csv(io.BytesIO(text),dtype=str)
        assert len(d.columns)==2 and d.columns[1]==series and d.columns[0] in ('DATE','observation_date')
        d.columns=['observation_date','value_string']
        d['observation_date']=pd.to_datetime(d.observation_date,errors='raise').dt.normalize()
        d['value']=pd.to_numeric(d.value_string.replace('.',np.nan),errors='coerce')
        assert d.observation_date.is_monotonic_increasing and d.observation_date.is_unique and len(d)>3000
        parsed[series]=d[['observation_date','value']].rename(columns={'value':series})
    paired=parsed[plan['requests'][0]['series']].merge(parsed[plan['requests'][1]['series']],on='observation_date',how='outer',validate='one_to_one').sort_values('observation_date')
    paired['spread_pctpoints']=paired[plan['requests'][0]['series']]-paired[plan['requests'][1]['series']]
    valid=paired.loc[np.isfinite(paired.spread_pctpoints)].copy()
    assert len(valid)>2500 and valid.observation_date.is_unique
    qlib.init(provider_uri=str(root/plan['provider']['calendar_uri']),region='us')
    cal=pd.DatetimeIndex(D.calendar(start_time='2010-12-01',end_time=plan['provider']['reused_held_signal_dates'][1],freq='day'))
    state['Qlib_D_calendar_calls']=1;atomic_json(state,card/'run_state.json')
    train=cal[(cal>=pd.Timestamp('2011-01-03'))&(cal<=pd.Timestamp('2022-12-30'))]
    held=cal[(cal>=pd.Timestamp('2023-01-03'))&(cal<=pd.Timestamp('2026-10-07'))]
    assert len(train)==plan['provider']['expected_train_rows'] and len(held)==plan['provider']['expected_held_rows']
    vdates=valid.observation_date.to_numpy('datetime64[ns]')
    vspread=valid.spread_pctpoints.to_numpy(float)
    rows=[]
    for i,day in enumerate(cal):
        status='insufficient_lag';obs=pd.NaT;value=np.nan;age=np.nan
        if i>=5:
            cutoff=cal[i-5]
            j=int(np.searchsorted(vdates,np.datetime64(cutoff),side='right')-1)
            if j>=0:
                obs=pd.Timestamp(vdates[j]);age=int(i-(cal.searchsorted(obs,side='right')-1))
                if age<=10:
                    status='known_retrospective_not_vintage_qualified';value=float(vspread[j])
                else:status='stale_over_10_sessions'
            else:status='no_paired_observation'
        rows.append({'signal_date':day,'source_observation_date':obs,'source_age_sessions':age,'CP_A2P2_minus_AA_30day_pctpoints':value,'status':status})
    mapped=pd.DataFrame(rows)
    known=mapped.status.eq('known_retrospective_not_vintage_qualified')
    t=mapped.signal_date.isin(train);h=mapped.signal_date.isin(held)
    train_known=int((known&t).sum());held_known=int((known&h).sum())
    pass_gate=train_known/len(train)>=plan['provider']['minimum_known_fraction_each'] and held_known/len(held)>=plan['provider']['minimum_known_fraction_each']
    assert len(mapped)==len(cal) and np.isfinite(mapped.loc[known,'CP_A2P2_minus_AA_30day_pctpoints']).all()
    summary={'experiment':plan['experiment'],'decision':'conditional_retrospective_Fed_CP_rating_spread_source_calendar_pass_first_vintage_pending_no_fit' if pass_gate else 'needs_Fed_CP_pair_coverage_no_fit','objective_achieved':False,'plan_sha256':state['plan_sha256'],'source_archive':'source_archive_receipt.json','retrieval':'retrieval_receipt.json','raw_rows_each':{k:len(v) for k,v in parsed.items()},'paired_finite_observations':len(valid),'train_calendar_days':len(train),'train_known_days':train_known,'held_calendar_days':len(held),'held_known_days':held_known,'known_fractions':{'train':train_known/len(train),'held':held_known/len(held)},'status_counts':{str(k):int(v) for k,v in mapped.status.value_counts().items()},'source_gate_passed':bool(pass_gate),'historical_first_public_vintage_qualified':False,'new_HTTP_GETs':2,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
    with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
        rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive=source_archive,original_csv_bytes={req['series']:(snap/(req['series']+'.csv')).read_bytes() for req in plan['requests']},response_headers={req['series']:(snap/(req['series']+'.headers.txt')).read_bytes() for req in plan['requests']})
        rec.save_objects(artifact_path='analysis',parsed=parsed,paired=paired,mapped=mapped)
        rec.log_metrics(model_fits=0,native_accounts=0,train_known_fraction=train_known/len(train),held_known_fraction=held_known/len(held))
        summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(card/'qlib_recorder'))
    atomic_json(summary,card/'summary.json')
    binding={'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'source_plan':'plan.json','ssh':settings['ssh']}
    atomic_json(binding,card/'nas_binding.json')
    evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
    remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
    summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True};atomic_json(summary,card/'summary.json')
    state.update(status='completed',new_Qlib_Recorders=1,recorder_id=summary['recorder_id']);atomic_json(state,card/'run_state.json')
    (card/'conclusion.md').write_text(
        '# Fed 30日商业票据等级利差：来源日历预检\n\n'
        f'按冻结两次GET接收A2/P2与AA非金融商业票据30日利率当前回溯CSV，原响应/报头共4份文件SHA、大小与NAS限定恢复核对。两个利率须同观察日均为数值才相减；不把缺价记零。\n\n'
        f'以真实Qlib交易日将同日等级利差保守滞后至少5会话，超过10会话未有成对报价则未知。2011–2022训练{len(train)}日中已知{train_known}，2023–2026复用评价{len(held)}日中已知{held_known}；全部日历位置保存。决定`{summary["decision"]}`。这只是当前回溯数据的来源/缺失门，不证明2011–2022首次发布字节、修订或历史盘前可用。\n\n'
        f'本卡不读SOXX未来收益、不拟合、不做账户。下一步若来源门通过，必须先限定原始联储历史发布版本或ALFRED版本，再登记与原Stage1 39项输入匹配的唯一信息检验；即使历史信息通过仍须完整0bp Tiered账户与v2门。Recorder `{summary["recorder_id"]}`，原始来源NAS SHA `{source_archive["archive_sha256"]}`，Recorder NAS SHA `{remote}`。\n'
    )
    print(summary['decision'],len(train),train_known,len(held),held_known,len(valid),summary['recorder_id'])


2026/10/23 04:45:25 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


needs_exact_Fed_CP_source_transport_no_fit [('RIFSPPNA2P2D30NB', 0, 0), ('RIFSPPNAAD30NB', 0, 0)]


## 中文结论

以 conclusion.md 和 summary.json 为准。来源通过不是因子有效或PIT/前向资格。